# 02 · ASR (T-one) + диаризация (pyannote)

Из `data/audio/<id>.wav` (16 кГц, mono, телефонный канал) получаем сегментированный транскрипт с ролями и таймстампами.

```
data/audio/<id>.wav
   -> T-one (Docker, WS /api/ws)   фразы: {text, start_time, end_time}
   -> pyannote-3.1 (local)         сегменты: {SPEAKER_x, start, end}
   -> слияние по пересечению       TranscriptSegment: {speaker, text, start, end}
   -> data/transcripts/<id>.json
```

Запуск контейнера:

```bash
docker pull tinkoffcreditsystems/t-one:0.1.0
docker run -d --name t-one -p 8080:8080 tinkoffcreditsystems/t-one:0.1.0
```

Протокол WS: сервер присылает `{"event":"ready"}` → отправить 16-bit PCM байты (8 кГц, mono) → ответ `{"event":"transcript","phrase":{"text","start_time","end_time"}}` → финализация пустыми байтами.

Нужен `.env` с `HF_TOKEN` (gated-модели pyannote: `pyannote/speaker-diarization-3.1`, `pyannote/segmentation-3.0`).

In [1]:
# 1. Импорты, настройки, пути
import json
import sys
from pathlib import Path

# Логика уже реализована в services/api (api.ml.*) — ноутбук вызывает её, а не дублирует код.
RESEARCH_DIR = Path.cwd()
SRC_DIR = RESEARCH_DIR.parent / "src"  # services/api/src
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Settings (.env) читаются относительно CWD — переходим в services/api, где конфиг находит корневой .env.
import os
os.chdir(RESEARCH_DIR.parent)

from app.ml.asr.tone import ToneASRClient
from app.ml.diarization.pyannote import PyannoteDiarizer
from app.ml.stt.pipeline import merge, read_phone_pcm, save_transcript
from app.ml.stt.schemas import Transcript
from app.config import get_settings
settings = get_settings()

AUDIO_DIR = RESEARCH_DIR / "data" / "audio"
GOLD_DIR = RESEARCH_DIR / "data" / "gold"
OUT_DIR = RESEARCH_DIR / "data" / "transcripts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ASR      :", settings.asr_ws_url)
print("HF_TOKEN :", "задан" if settings.hf_token else "НЕ задан")
print("AUDIO_DIR:", AUDIO_DIR)

ASR      : ws://sofiya-mac:8080/api/ws
HF_TOKEN : задан
AUDIO_DIR: /Users/sofiya/Projects/MTS_Hackathon/services/api/research/data/audio


In [2]:
# 2. Подготовка аудио: 16кГц wav -> 8кГц int16 PCM (для WS контейнера)
CALL_ID = "explicit_deadline_01"

wav_path = AUDIO_DIR / f"{CALL_ID}.wav"
pcm = read_phone_pcm(wav_path)  # resample 16кГц -> 8кГц, mono int16
duration = len(pcm) // 2 / 8000

print(f"{wav_path.name}: PCM={len(pcm)} байт (~{duration:.1f} c, 8кГц int16 mono)")

explicit_deadline_01.wav: PCM=275600 байт (~17.2 c, 8кГц int16 mono)


In [3]:
# 3. ASR через WebSocket T-one: фразы с таймстампами
asr = ToneASRClient(settings.asr_ws_url)
phrases = await asr.transcribe(pcm)

for i, phrase in enumerate(phrases):
    print(f"[{phrase.start_time:6.2f}-{phrase.end_time:6.2f}] {phrase.text}")
print(f"фраз: {len(phrases)}")

[  0.39-  6.06] здравствуйте игорь владимирович я анна из клиники восстановления по поводу вашего курса реабилитации
[  6.72-  9.99] здравствуйте анна да я вас внимательно слушаю
[ 10.65- 14.46] мы подготовили для вас договор я пришлю его до пятницы удобным вам способом
[ 15.06- 16.89] хорошо буду ждать спасибо
фраз: 4


In [4]:
# 4. Диаризация pyannote: спикеры по времени
diarizer = PyannoteDiarizer(settings.hf_token) if settings.hf_token else None
speaker_segments = diarizer.diarize(wav_path) if diarizer else []

for segment in speaker_segments:
    print(f"[{segment.start:6.2f}-{segment.end:6.2f}] {segment.speaker}")
print(f"сегментов: {len(speaker_segments)}")

/Users/sofiya/Projects/MTS_Hackathon/services/api/research/.venv/lib/python3.13/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1913.)
  std = sequences.std(dim=-1, correction=1)


[  0.33-  2.28] SPEAKER_01
[  2.41-  6.22] SPEAKER_01
[  6.83- 10.19] SPEAKER_00
[ 10.65- 14.51] SPEAKER_01
[ 14.97- 17.14] SPEAKER_00
сегментов: 5


In [5]:
# 5. Слияние: каждой фразе спикера по max-пересечению, роли operator/client
merged = merge(phrases, speaker_segments)

for segment in merged:
    print(
        f"[{segment.start:6.2f}-{segment.end:6.2f}] {segment.speaker:9s} {segment.text}"
    )

[  0.39-  6.06] operator  здравствуйте игорь владимирович я анна из клиники восстановления по поводу вашего курса реабилитации
[  6.72-  9.99] client    здравствуйте анна да я вас внимательно слушаю
[ 10.65- 14.46] operator  мы подготовили для вас договор я пришлю его до пятницы удобным вам способом
[ 15.06- 16.89] client    хорошо буду ждать спасибо


In [6]:
# 6. WER vs gold (normalize: lower, без пунктуации, ё->е)
import re


def normalize(text: str) -> str:
    text = text.lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.split())


def word_error_rate(reference: str, hypothesis: str) -> float:
    ref = normalize(reference).split()
    hyp = normalize(hypothesis).split()
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + cost)
    return d[len(ref)][len(hyp)] / max(1, len(ref))


gold = json.loads((GOLD_DIR / f"{CALL_ID}.json").read_text(encoding="utf-8"))
gold_texts = [turn["text"] for turn in gold["dialogue"] if turn["text"].strip()]
hyp_texts = [segment.text for segment in merged]

rows = []
for i, (ref, hyp) in enumerate(zip(gold_texts, hyp_texts)):
    rows.append((i, ref, hyp, word_error_rate(ref, hyp)))

for i, ref, hyp, wer_val in rows:
    print(f"turn {i}: WER={wer_val:.2f}  ref={ref!r}\n      hyp={hyp!r}")

if rows:
    avg_wer = sum(r[3] for r in rows) / len(rows)
    print(f"средний WER по {len(rows)} репликам: {avg_wer:.2f}")

turn 0: WER=0.08  ref='Здравствуйте, Игорь Владимирович! Я Анна из клиники «Восстановление», по поводу вашего курса реабилитации.'
      hyp='здравствуйте игорь владимирович я анна из клиники восстановления по поводу вашего курса реабилитации'
turn 1: WER=0.00  ref='Здравствуйте, Анна. Да, я вас внимательно слушаю.'
      hyp='здравствуйте анна да я вас внимательно слушаю'
turn 2: WER=0.00  ref='Мы подготовили для вас договор, я пришлю его до пятницы удобным вам способом.'
      hyp='мы подготовили для вас договор я пришлю его до пятницы удобным вам способом'
turn 3: WER=0.00  ref='Хорошо, буду ждать. Спасибо!'
      hyp='хорошо буду ждать спасибо'
средний WER по 4 репликам: 0.02


In [7]:
# 7. Сохранение data/transcripts/<id>.json + таблица-сводка
transcript = Transcript(conversation_id=CALL_ID, segments=merged)
out = save_transcript(transcript, OUT_DIR)
print("Сохранено:", out)

print(f"{'start':>6} {'end':>6} {'speaker':<9} text")
for segment in transcript.segments:
    print(
        f"{segment.start:6.2f} {segment.end:6.2f} {segment.speaker:<9} {segment.text}"
    )

Сохранено: /Users/sofiya/Projects/MTS_Hackathon/services/api/research/data/transcripts/explicit_deadline_01.json
 start    end speaker   text
  0.39   6.06 operator  здравствуйте игорь владимирович я анна из клиники восстановления по поводу вашего курса реабилитации
  6.72   9.99 client    здравствуйте анна да я вас внимательно слушаю
 10.65  14.46 operator  мы подготовили для вас договор я пришлю его до пятницы удобным вам способом
 15.06  16.89 client    хорошо буду ждать спасибо


Smoke-тест одного звонка (`explicit_deadline_01.wav`, 17.2 c):
- фраз не 0, text непустые;
- start_time/end_time внутри длительности звонка;
- sлияние даёт спикеров operator + client.

In [8]:
# Smoke-тест: phrases non-empty, timestamps внутри длительности, спикеры operator/client
assert len(transcript.segments) > 0, "фраз не должно быть 0"
assert all(segment.text for segment in transcript.segments), (
    "текст не должен быть пустым"
)
assert all(
    0 <= segment.start <= segment.end <= duration + 0.01
    for segment in transcript.segments
), "таймкоды вне длительности"
speakers = {segment.speaker for segment in transcript.segments}
assert {"operator", "client"} <= speakers, f"не хватает ролей: {speakers}"

print(f"фраз: {len(transcript.segments)}; спикеры: {sorted(speakers)}")
print("Smoke-тест пройден OK")

фраз: 4; спикеры: ['client', 'operator']
Smoke-тест пройден OK
